In [1]:
import os
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

from glob import glob
from tqdm import tqdm_notebook
from astroquery.jplsbdb import SBDB

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
all_aster = pd.read_csv("../data/jpl/jpl_asteroids_names_all_20251215.csv")
all_aster["Object name"] = [x.strip() for x in all_aster["full_name"].values]
all_aster

,spkid,full_name,pdes,name,prefix,Object name
0,20000001,1 Ceres (A801 AA),1,Ceres,NaN,1 Ceres (A801 AA)
1,20000002,2 Pallas (A802 FA),2,Pallas,NaN,2 Pallas (A802 FA)
2,20000003,3 Juno (A804 RA),3,Juno,NaN,3 Juno (A804 RA)
3,20000004,4 Vesta (A807 FA),4,Vesta,NaN,4 Vesta (A807 FA)
4,20000005,5 Astraea (A845 XA),5,Astraea,NaN,5 Astraea (A845 XA)
...,...,...,...,...,...,...
1482897,3246764,(4847 P-L),4847 P-L,NaN,NaN,(4847 P-L)
1482898,3246834,(6331 P-L),6331 P-L,NaN,NaN,(6331 P-L)
1482899,3013075,(6344 P-L),6344 P-L,NaN,NaN,(6344 P-L)
1482900,54539114,(2015 BA620),2015 BA620,NaN,NaN,(2015 BA620)


In [35]:
sectors = np.arange(101, 108)
for sec in tqdm_notebook(sectors, total=len(sectors), desc=f"Working on Sector"):
    print(f"Sector {sec}")
    sec_catalog = pd.concat([pd.read_csv(x, index_col=0) for x in glob(f"../data/jpl/jpl_small_bodies_tess_s{sec:04}-*-*_catalog.csv")], 
                            axis=0
                      ).drop_duplicates(subset=["Object name"]).reset_index(drop=True)
    sec_catalog["track_name"] = [x.replace(' ', '-').replace('/', '-') for x in sec_catalog['id'].values]
    tfiles = glob(f"../data/jpl/tracks/sector{sec:04}/tess-ffi_s{sec:04}-0-0_*_*res.feather")
    aster_with_tracks = []
    for ff in tqdm_notebook(tfiles, total=len(tfiles), desc="Crowling lc files", leave=False):
        org_track = pd.read_feather(ff)
        cameras = org_track.camera.unique()
        ccds = org_track.ccd.unique()
        trks = [[ca, cc] for cc in ccds for ca in cameras]
        tracks = []
        for camera, ccd in trks:
            track = org_track.query(f"camera == {camera} and ccd == {ccd}")
            if len(track) > 0:
                tracks.append(track)
    
        for trk in tracks:
            row = pd.Series([ 
                            sec,
                            trk.camera.unique()[0],
                            trk.ccd.unique()[0],
                            os.path.basename(ff),
                            trk.vmag.mean(),
                            trk.time.iloc[-1] - trk.time.iloc[0],
                            np.hypot(trk.column.iloc[-1] - trk.column.iloc[0], trk.row.iloc[-1] - trk.row.iloc[0]),
                            np.abs((trk.row.iloc[-1] - trk.row.iloc[0]) / (trk.time.iloc[-1] - trk.time.iloc[0])),
                            np.abs((trk.column.iloc[-1] - trk.column.iloc[0]) / (trk.time.iloc[-1] - trk.time.iloc[0])),
                            trk.time.min(),
                            trk.time.max(),
                            ], 
                            index=[
                                "sector",
                                "camera",
                                "ccd",
                                "file_name",
                                "vmag",
                                "time arc (d)",
                                "pixel arc",
                                "row rate (pix/d)",
                                "column rate (pix/d)", 
                                "tstart",
                                "tstop",
                            ], 
                            name=os.path.basename(ff).split("_")[2]
                           )
            aster_with_tracks.append(row)
        # break
        
    aster_with_tracks = pd.concat(aster_with_tracks, axis=1).T.reset_index().rename(columns={"index": "track_name"})
    aster_with_tracks = pd.merge(aster_with_tracks, sec_catalog, on="track_name", how="left")
    aster_with_tracks["pixel rate (pix/d)"] = aster_with_tracks["pixel arc"] / aster_with_tracks["time arc (d)"]
    
    cols = [
        'Object name', 'name', 'id', 'track_name',
        'sector', 'camera', 'ccd', 'file_name', 
        'vmag', 'kind',
        'time arc (d)', 'pixel arc', 
        'tstart', 'tstop',
        'row rate (pix/d)', 'column rate (pix/d)', 
        'pixel rate (pix/d)', 'RA rate ("/h)', 'Dec rate ("/h)',
        'H_mag', 'Eccentricity', 'Inclination (deg)', "Perihelion (au)",
    ]
    
    aster_with_tracks = aster_with_tracks.loc[:, cols]
    aster_with_tracks_save = pd.merge(aster_with_tracks, all_aster.loc[:, ["spkid", "pdes", "name", "Object name"]], on="Object name", how="left")
    
    # fix spkid by quering SBDB using ID
    nan_spkids = aster_with_tracks_save[aster_with_tracks_save.spkid.isna()]
    for k, row in tqdm_notebook(nan_spkids.iterrows(), total=len(nan_spkids), desc="Fixing nan spkid", leave=False):
        obj_id = row["id"]
        if pd.isna(obj_id):
            obj_id = row["track_name"].replace("-", " ")
        sbdb = SBDB.query(obj_id)
        try:
            aster_with_tracks_save.loc[k, "spkid"] = sbdb["object"]["spkid"]
            aster_with_tracks_save.loc[k, "Object name"] = sbdb["object"]["fullname"]
        except:
            continue
    
    # fix spkid by quering SBDB using name for remaining rows
    nan_spkids = aster_with_tracks_save[aster_with_tracks_save.spkid.isna()]
    for k, row in tqdm_notebook(nan_spkids.iterrows(), total=len(nan_spkids), desc="Fixing remaining nan spkid", leave=False):
        obj_id = row["name_x"]
        if pd.isna(obj_id):
            obj_id = row["track_name"].replace("-", " ")
        sbdb = SBDB.query(obj_id)
        try:
            aster_with_tracks_save.loc[k, "spkid"] = sbdb["object"]["spkid"]
            aster_with_tracks_save.loc[k, "Object name"] = sbdb["object"]["fullname"]
        except:
            continue
    aster_with_tracks_save = aster_with_tracks_save.drop(aster_with_tracks_save[aster_with_tracks_save.spkid.isna()].index)
    
    # fix kind
    nan_spkids = aster_with_tracks_save[aster_with_tracks_save.kind.isna()]
    for k, row in tqdm_notebook(nan_spkids.iterrows(), total=len(nan_spkids), desc="Fixing missing kind", leave=False):
        obj_id = row["name_x"]
        if pd.isna(obj_id):
            obj_id = row["track_name"].replace("-", " ")
        sbdb = SBDB.query(obj_id)
        try:
            aster_with_tracks_save.loc[k, "kind"] = sbdb["object"]["kind"][0]
        except:
            continue
    
    # fix id 
    nan_spkids = aster_with_tracks_save[aster_with_tracks_save.id.isna()]
    for k, row in tqdm_notebook(nan_spkids.iterrows(), total=len(nan_spkids), desc="Fixing missing ids", leave=False):
        aster_with_tracks_save.loc[k, "id"] = row["track_name"].replace("-", " ")
    
    cols = [
        'spkid', 'Object name', 'pdes', 'name_y', 'track_name',
        'sector', 'camera', 'ccd', 'file_name', 
        'vmag', 'kind',
        'time arc (d)', 'pixel arc', 
        'tstart', 'tstop',
        'row rate (pix/d)', 'column rate (pix/d)', 
        'pixel rate (pix/d)', 'RA rate ("/h)', 'Dec rate ("/h)',
        'H_mag', 'Eccentricity', 'Inclination (deg)', "Perihelion (au)",
    ]
    
    aster_with_tracks_format = aster_with_tracks_save[cols].rename(
        columns={
            "name_y": "IAU name", 
            # "id": "Principal desig",
            "vmag": "Visual magnitude",
            "H_mag": "Absolute magntiude (H)",
            "spkid": "SPK ID",
        }
    )
    aster_with_tracks_format = aster_with_tracks_format.sort_values("Visual magnitude").reset_index(drop=True)
    
    aster_with_tracks_format["sector"] = aster_with_tracks_format["sector"].astype(int)
    aster_with_tracks_format["camera"] = aster_with_tracks_format["camera"].astype(int)
    aster_with_tracks_format["ccd"] = aster_with_tracks_format["ccd"].astype(int)
    aster_with_tracks_format["SPK ID"] = aster_with_tracks_format["SPK ID"].astype(int)
    aster_with_tracks_format["Visual magnitude"] = aster_with_tracks_format["Visual magnitude"].astype(float)
    aster_with_tracks_format["time arc (d)"] = aster_with_tracks_format["time arc (d)"].astype(float)
    aster_with_tracks_format["tstart (jd)"] = aster_with_tracks_format["tstart"].astype(float)
    aster_with_tracks_format["tstop (jd)"] = aster_with_tracks_format["tstop"].astype(float)
    aster_with_tracks_format["pixel arc"] = aster_with_tracks_format["pixel arc"].astype(float)
    aster_with_tracks_format["row rate (pix/d)"] = aster_with_tracks_format["row rate (pix/d)"].astype(float)
    aster_with_tracks_format["column rate (pix/d)"] = aster_with_tracks_format["column rate (pix/d)"].astype(float)
    aster_with_tracks_format["pixel rate (pix/d)"] = aster_with_tracks_format["pixel rate (pix/d)"].astype(float)
    aster_with_tracks_format = aster_with_tracks_format.drop_duplicates(subset=["SPK ID", "Object name"], keep="last")
    
    aster_with_tracks_format.to_csv(f"../data/jpl/tess_observ/tess_observ_asteroid_lookup_table_sec{sec:04}.csv")
    
    # break

Working on Sector:   0%|          | 0/7 [00:00<?, ?it/s]

Sector 101


Crowling lc files:   0%|          | 0/959 [00:00<?, ?it/s]

Fixing nan spkid:   0%|          | 0/12 [00:00<?, ?it/s]

Fixing remaining nan spkid: 0it [00:00, ?it/s]

...

Crowling lc files:   0%|          | 0/1573 [00:00<?, ?it/s]

Fixing nan spkid:   0%|          | 0/4 [00:00<?, ?it/s]

Fixing remaining nan spkid:   0%|          | 0/4 [00:00<?, ?it/s]

Fixing missing kind: 0it [00:00, ?it/s]

Fixing missing ids: 0it [00:00, ?it/s]

In [15]:
sec

101

In [18]:
from tesswcs import pointings

In [31]:
row = pointings.to_pandas().query("Sector == 101")
print(row)
time = np.arange(row["Start"].values[0], row["End"].values[0] + 0.1, 0.5) - 2457000
time

     Cycle  Sector        RA      Dec      Roll      Start        End
100      8     101  209.0229 -62.0323  217.7538  2461100.5  2461126.5


array([4100.5, 4101. , 4101.5, 4102. , 4102.5, 4103. , 4103.5, 4104. ,
       4104.5, 4105. , 4105.5, 4106. , 4106.5, 4107. , 4107.5, 4108. ,
       4108.5, 4109. , 4109.5, 4110. , 4110.5, 4111. , 4111.5, 4112. ,
       4112.5, 4113. , 4113.5, 4114. , 4114.5, 4115. , 4115.5, 4116. ,
       4116.5, 4117. , 4117.5, 4118. , 4118.5, 4119. , 4119.5, 4120. ,
       4120.5, 4121. , 4121.5, 4122. , 4122.5, 4123. , 4123.5, 4124. ,
       4124.5, 4125. , 4125.5, 4126. , 4126.5])

In [28]:
time.shape

(26,)

In [22]:
from tesscube import TESSCube

In [23]:
TESSCube(sector=90, camera=1, ccd=1).time

array([3747.15951787, 3747.16183272, 3747.16414756, ..., 3775.0759389 ,
       3775.07825365, 3775.08056841])

In [33]:
import requests

In [34]:
requests.exceptions.HTTPError

requests.exceptions.HTTPError